# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets:")
record_sets = dataset.metadata.recordSet
if not isinstance(record_sets, list):
    record_sets = [record_sets]
for rs in record_sets:
    print(f"  - RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("    Fields (by @id):")
    for field in fields:
        print(f"      - {field['@id']} ({field.get('name', '(no name)')})")

## 3. Data Extraction
Load data from all available record sets into DataFrames. Each record set and field is referenced by its `@id`.

In [ ]:
# Collect all record set @ids
if not dataset.metadata.recordSet:
    print("No record sets found in schema. Some Croissant packages may define them in a top-level 'distribution' field.")
    # Try an alternate approach
    record_sets_ids = []
    if getattr(dataset.metadata, 'distribution', None):
        for dist in dataset.metadata.distribution:
            if '@id' in dist:
                record_sets_ids.append(dist['@id'])
    print(f"Found possible record sets via 'distribution': {record_sets_ids}")
else:
    record_sets_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

# Try to extract data from the first discovered record set
dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
    except Exception as e:
        print(f"Failed loading records for {record_set_id}: {e}")

# Example: show columns from the first successfully loaded DataFrame
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Data columns for record set '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded. Please check dataset resources or definition.")

## 4. Exploratory Data Analysis (EDA)
Basic EDA: Filter records, normalize a numerical field, and group by a categorical field.
Use the `@id` of the fields discovered above. Adjust field choices below for your specific dataset.

In [ ]:
import numpy as np

# For demonstration, pick main_record_set_id and print out current columns
if dataframes:
    df = dataframes[main_record_set_id]
    print(f"Fields/columns available: {df.columns.tolist()}")
    # Try to pick a numeric field present in the dataframe
    # (update these depending on inspection of the data)
    sample_numeric_fields = ['cr:PeptideCounts', 'cr:MW', 'cr:coverage']
    numeric_field = None
    for field in sample_numeric_fields:
        if field in df.columns:
            numeric_field = field
            break
    if not numeric_field:
        print("No candidate numeric field found in columns for demonstration. Please check fields above.")
    else:
        # Filter: Show all rows where the numeric field > threshold (here, threshold=10)
        threshold = 10
        filtered_df = df[df[numeric_field].apply(pd.to_numeric, errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} found")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce') - filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce').mean()
        ) / filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce').std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by example field if present
        group_field_candidates = ['cr:description', 'cr:accession', 'cr:Sample']
        group_field = None
        for c in group_field_candidates:
            if c in df.columns:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field found to group by in this demo.")
else:
    print("No dataframe loaded for EDA.")

## 5. Visualization
Visualize value distributions or relationships between two fields, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric_field if available
if dataframes and numeric_field:
    data = df[numeric_field].apply(pd.to_numeric, errors='coerce').dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(data, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Example: scatterplot between two numeric fields, if available
    numeric_candidates = [c for c in df.columns if c != numeric_field]
    scatter_field = None
    for nc in numeric_candidates:
        if pd.api.types.is_numeric_dtype(df[nc]) or pd.to_numeric(df[nc], errors='coerce').notna().any():
            scatter_field = nc
            break
    if scatter_field:
        plt.figure(figsize=(7,5))
        plt.scatter(df[numeric_field].apply(pd.to_numeric, errors='coerce'),
                    df[scatter_field].apply(pd.to_numeric, errors='coerce'))
        plt.xlabel(numeric_field)
        plt.ylabel(scatter_field)
        plt.title(f"Scatterplot of {numeric_field} vs. {scatter_field}")
        plt.show()
    else:
        print("No second numeric field found for scatterplot.")
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and perform basic analysis on a Croissant-formatted biological dataset using `mlcroissant`.

**Key Learnings:**
- The dataset describes mass spectrometry results for human extracellular vesicle proteins with rich, machine-readable metadata.
- We reviewed available record sets and fields referenced by their `@id`.
- Example data extraction, filtering, normalization, grouping, and visualization steps were shown, referencing all entities by their `@id` field for reproducibility.

For deeper analysis, customize field and record set choices above; inspect available metadata fields for further scientific exploration.